In [1]:
import json
import torch.nn as nn
import torch
from model.Transformer import Transformer,encoder_stack,decoder_stack
from model.Transformer_block import encoder_block,decoder_block
from Data.dataset import Dataset

with open("src_vocab.json", "r") as f:
    src_vocab = json.load(f)

with open("tgt_vocab.json", "r") as f:
    tgt_vocab = json.load(f)

embed_dim = 64

encoder_blocks= [encoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]
decoder_blocks= [decoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]

enc_stack= encoder_stack(encoder_blocks)
dec_stack= decoder_stack(decoder_blocks)

model= Transformer(enc_stack, dec_stack, embed_dim, len(src_vocab), len(tgt_vocab))

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= model.to(device)

In [4]:
from datasets import load_dataset
import re
from torch.utils.data import DataLoader
from Data.dataloader import collate
from Data.vocab import clean

ds = load_dataset("Helsinki-NLP/opus-100", "en-it")

data_pair = []

for sample in ds["train"]:
    en= clean(sample["translation"]["en"])
    it= clean(sample["translation"]["it"])

    data_pair.append((en, it))


train_dataset= Dataset(data_pair[0:75000],src_vocab_path="src_vocab.json",tgt_vocab_path="tgt_vocab.json", src_vocab=src_vocab,tgt_vocab=tgt_vocab)
val_dataset= Dataset(data_pair[-10000:],src_vocab_path="src_vocab.json",tgt_vocab_path="tgt_vocab.json", src_vocab=src_vocab,tgt_vocab=tgt_vocab)


train_loader= DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate)
val_loader= DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate)



In [ ]:
scaler= torch.amp.GradScaler("cuda")

epochs=10
criterion= torch.nn.CrossEntropyLoss(ignore_index=0)
optimizer= torch.optim.AdamW(model.parameters(), lr=1e-4)

accumulation_steps=4
best_loss= float("inf")

for epoch in range(epochs):

    running_train_loss=0
    running_val_loss=0
    model.train()

    optimizer.zero_grad()

    for i,batch in enumerate(train_loader):

        src = batch["src"].to(device)
        tgt_in = batch["tgt_in"].to(device)
        tgt_out = batch["tgt_out"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):

            output= model(src,tgt_in)
            loss= criterion(output.reshape(-1,len(tgt_vocab)),tgt_out.reshape(-1))


        loss= loss/accumulation_steps
        scaler.scale(loss).backward()

        if (i+1)%accumulation_steps==0:

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad() 

        running_train_loss += loss.item() * accumulation_steps

    if (i + 1) % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_train_loss = running_train_loss / len(train_loader)

    for batch in val_loader:

        src = batch["src"].to(device)
        tgt_in = batch["tgt_in"].to(device)
        tgt_out = batch["tgt_out"].to(device)

        output= model(src,tgt_in)
        val_loss= criterion(output.reshape(-1,len(tgt_vocab)),tgt_out.reshape(-1))

        running_val_loss += val_loss.item()
    

    avg_val_loss = running_val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{epochs}] | Training Loss: {avg_train_loss:.4f}")
    print(f"Epoch [{epoch+1}/{epochs}] | Validation Loss: {avg_val_loss:.4f}")






/home/aayam/Torch_Transformer/model/Transformer_block.py:18: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /opt/conda/conda-bld/pytorch_1729647429097/work/aten/src/ATen/native/ReduceOps.cpp:1823.)
  var= x.var(dim=-1, keepdim=True, unbiased=False)


Epoch [1/10] | Loss: 7.1446
Epoch [2/10] | Loss: 6.1760
Epoch [3/10] | Loss: 5.7769
Epoch [4/10] | Loss: 5.5155
Epoch [5/10] | Loss: 5.3202
Epoch [6/10] | Loss: 5.1586
Epoch [7/10] | Loss: 5.0277
Epoch [8/10] | Loss: 4.9050
Epoch [9/10] | Loss: 4.8001
Epoch [10/10] | Loss: 4.7142


# Current status
gradient accumulation used to make my model update weights per 8 batches . The loss is clearly going downwards but at a slower rate . Epochs need to be increased .